In [ ]:
# Configuration environnement Colab/Local
import sys, os, pathlib
import warnings
warnings.filterwarnings('ignore')

# Détection automatique
if 'google.colab' in str(get_ipython()):
    PROJECT_ROOT = pathlib.Path("/content/projet_ai")
    %cd /content/projet_ai
else:
    PROJECT_ROOT = pathlib.Path.cwd().parent if 'notebooks' in str(pathlib.Path.cwd()) else pathlib.Path.cwd()

# Ajouter src/ au path
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Créer dossiers de sortie
for folder in ["reports/figures", "reports/tables", "models"]:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

# Fixer seeds
import numpy as np
np.random.seed(42)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"Dossiers de sortie créés")

# Étape 4 : Analyse SHAP (interprétabilité)

## Meilleur modèle : XGBoost

TreeExplainer sur sous-échantillon test (5000 points).

---


## 0. Configuration


In [ ]:
import os
import sys
import time
import warnings
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.loader import load_aps_data, get_X_y, compute_total_cost, COST_FP, COST_FN
from src.utils.reproducibility import set_all_seeds, log_environment_info, setup_logging, get_cv_splitter
from src.evaluation.metrics import compute_all_metrics, print_metrics_report, find_optimal_threshold
from src.evaluation.plots import setup_plot_style, save_figure, COLORS

warnings.filterwarnings('ignore')
setup_logging(level=logging.INFO)
setup_plot_style()
%matplotlib inline

SEED = 42
set_all_seeds(SEED)
N_FOLDS = 5
cv = get_cv_splitter(n_splits=N_FOLDS, seed=SEED)

FIGURES_DIR = os.path.join(PROJECT_ROOT, 'reports', 'figures')
TABLES_DIR = os.path.join(PROJECT_ROOT, 'reports', 'tables')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')
for d in [FIGURES_DIR, TABLES_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
log_environment_info()
pd.show_versions()

import joblib
import shap
import xgboost as xgb

train_df, test_df = load_aps_data(project_root=PROJECT_ROOT)
X_train, y_train = get_X_y(train_df)
X_test, y_test = get_X_y(test_df)

xgb_pack = joblib.load(os.path.join(MODELS_DIR, 'xgboost_best.pkl'))
if not os.path.exists(os.path.join(MODELS_DIR, 'xgboost_best.pkl')):
    raise FileNotFoundError('Exécutez d\'abord 02_model_training.ipynb')

booster = xgb_pack['booster']
imputer = xgb_pack['imputer']


## 1. SHAP — valeurs globales et locales


In [ ]:
SHAP_SAMPLE = 5000
rng = np.random.RandomState(SEED)
idx = rng.choice(len(X_test), size=min(SHAP_SAMPLE, len(X_test)), replace=False)
X_shap = X_test.iloc[idx]
X_shap_imp = imputer.transform(X_shap)
y_shap = y_test.iloc[idx].values

print(f'Calcul SHAP sur {len(X_shap)} points...')
explainer = shap.TreeExplainer(booster)
shap_values = explainer.shap_values(xgb.DMatrix(X_shap_imp))

# Summary beeswarm
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_shap, show=False, max_display=20)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'shap_summary.png'), dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_shap, plot_type='bar', show=False, max_display=15)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'shap_summary_bar.png'), dpi=300, bbox_inches='tight')
plt.show()

mean_abs = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    'feature': X_shap.columns,
    'mean_abs_shap': mean_abs,
}).sort_values('mean_abs_shap', ascending=False)
top10 = importance_df.head(10)
display(top10)


In [ ]:
# Dependence plots — top 5
top5 = importance_df.head(5)['feature'].tolist()
for feat in top5:
    plt.figure(figsize=(8, 5))
    shap.dependence_plot(feat, shap_values, X_shap, show=False)
    plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'shap_dependence.png'), dpi=300, bbox_inches='tight')
plt.show()

# Prédictions pour exemples locaux
proba = booster.predict(xgb.DMatrix(X_shap_imp))
pred = (proba >= 0.5).astype(int)
tp_idx = np.where((y_shap == 1) & (pred == 1))[0]
fp_idx = np.where((y_shap == 0) & (pred == 1))[0]
fn_idx = np.where((y_shap == 1) & (pred == 0))[0]

examples = {}
if len(tp_idx): examples['TP'] = tp_idx[0]
if len(fp_idx): examples['FP'] = fp_idx[0]
if len(fn_idx): examples['FN'] = fn_idx[0]

for label, i in examples.items():
  plt.figure(figsize=(10, 4))
  shap.waterfall_plot(
      shap.Explanation(values=shap_values[i], base_values=explainer.expected_value,
                       data=X_shap.iloc[i].values, feature_names=X_shap.columns.tolist()),
      show=False, max_display=12,
  )
  plt.title(f'Waterfall — {label} (index {i})')
  plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'shap_waterfall_examples.png'), dpi=300, bbox_inches='tight')
plt.show()


## 2. Stabilité SHAP sur 5 folds


In [ ]:
print('Stabilité SHAP par fold (sous-échantillon 2000/fold)...')
rng = np.random.RandomState(SEED)
fold_importances = []
for fold, (tr, val) in enumerate(tqdm(cv.split(X_train, y_train), total=N_FOLDS)):
    X_v = X_train.iloc[val]
    y_v = y_train.iloc[val]
    n_s = min(2000, len(X_v))
    sub = rng.choice(len(X_v), n_s, replace=False)
    X_imp = imputer.transform(X_v.iloc[sub])
    # Réentraîner booster léger par fold serait coûteux — on utilise le modèle final
    # comme proxy de stabilité des explications sur sous-populations
    sv = explainer.shap_values(xgb.DMatrix(X_imp))
    fold_importances.append(np.abs(sv).mean(axis=0))

stab = np.vstack(fold_importances)
importance_df['shap_std_across_folds'] = stab.std(axis=0)
importance_df['stable'] = importance_df['shap_std_across_folds'] < importance_df['mean_abs_shap'].median()
display(importance_df.head(15))


## 3. Interprétation métier


In [ ]:
print('''
**Classe positive = panne APS** :
- SHAP > 0 pousse vers la prédiction « panne ».
- Capteurs top = candidats pour surveillance préventive.

**Biais / proxies** : vérifier si des capteurs corrélés à des conditions externes
(température, régime moteur) dominent sans lien causal APS.

**Actions** : prioriser maintenance sur les capteurs du top 10 SHAP stable.
''')

importance_df.head(10).to_csv(os.path.join(TABLES_DIR, 'shap_feature_importance.csv'), index=False)
print('Export : reports/tables/shap_feature_importance.csv')
print('Figures : shap_summary.png, shap_dependence.png, shap_waterfall_examples.png')
